# SpaceX Falcon 9 First-Stage Landing Prediction
**Student:** Malvika Pathak  
**Submission date:** 30 October 2026  
**Course:** IBM Applied Data Science Capstone

## Project objective
Explore historical Falcon 9 launch data and build classification models to estimate first-stage landing success. This notebook is designed to be executed in Jupyter/IBM Skills Network. It downloads the course dataset, produces the analysis outputs, and saves figures for the final presentation.

> **Important:** Run every cell from top to bottom before submitting. Check the output and adapt the discussion to the results actually produced in your environment. Do not submit an unexecuted notebook or claim results that you have not verified.


## 1. Imports and reproducibility
Install missing packages in your environment if needed. The core analysis uses pandas, NumPy, Matplotlib, Seaborn, scikit-learn, and requests.

In [ ]:
import warnings, os, json, requests
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

SEED = 42
os.makedirs("figures", exist_ok=True)
pd.set_option("display.max_columns", None)
sns.set_theme()


## 2. Load the course dataset
The IBM Skills Network dataset is preferred because the grading rubric is aligned to the course labs. If your course provides a local `dataset_part_2.csv`, place it beside the notebook and use that file first.

In [ ]:
DATA_URLS = [
    "dataset_part_2.csv",
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv",
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/dataset_part_2.csv"
]
df = None
errors = []
for source in DATA_URLS:
    try:
        candidate = pd.read_csv(source)
        if len(candidate) > 0:
            df = candidate.copy()
            print("Loaded dataset from:", source)
            break
    except Exception as exc:
        errors.append(f"{source}: {exc}")
if df is None:
    print("Could not download the course dataset automatically.")
    print("Upload dataset_part_2.csv to the notebook working directory, then rerun this cell.")
    print("\nAttempt details:")
    print("\n".join(errors))
else:
    print("Dataset shape:", df.shape)
    display(df.head())


## 3. Data quality and wrangling

In [ ]:
if df is not None:
    print("Rows, columns:", df.shape)
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean()*100).round(2),
        "unique_values": df.nunique(dropna=True)
    }))
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    if "PayloadMass" in df.columns:
        df["PayloadMass"] = pd.to_numeric(df["PayloadMass"], errors="coerce")
        df["PayloadMass"] = df["PayloadMass"].fillna(df["PayloadMass"].mean())
    if "Class" in df.columns:
        df["Class"] = pd.to_numeric(df["Class"], errors="coerce")
        df = df.dropna(subset=["Class"])
        df["Class"] = df["Class"].astype(int)
    print("Missing values after prescribed handling:")
    display(df.isna().sum().to_frame("missing_count"))


## 4. Exploratory data analysis

In [ ]:
if df is not None:
    if "Class" in df.columns:
        print("Landing class counts (0 = unsuccessful, 1 = successful as encoded by course):")
        display(df["Class"].value_counts().sort_index().rename_axis("Class").to_frame("Launches"))
        ax = sns.countplot(data=df, x="Class")
        ax.set(title="Landing outcome class counts", xlabel="Landing class", ylabel="Number of launches")
        plt.tight_layout(); plt.savefig("figures/landing_class_counts.png", dpi=180); plt.show()
    if "LaunchSite" in df.columns and "Class" in df.columns:
        rate = df.groupby("LaunchSite", dropna=False)["Class"].agg(["count","mean"]).sort_values("mean", ascending=False)
        rate = rate.rename(columns={"count":"Launches","mean":"Observed success proportion"})
        display(rate)
        ax = sns.barplot(data=df, x="LaunchSite", y="Class", errorbar=None)
        ax.set(title="Observed landing success proportion by launch site", xlabel="Launch site", ylabel="Mean landing class")
        ax.tick_params(axis="x", rotation=25); plt.tight_layout()
        plt.savefig("figures/success_by_launch_site.png", dpi=180); plt.show()
    if "Orbit" in df.columns and "Class" in df.columns:
        orbit_rate = df.groupby("Orbit")["Class"].agg(["count","mean"]).sort_values("mean", ascending=False)
        orbit_rate = orbit_rate.rename(columns={"count":"Launches","mean":"Observed success proportion"})
        display(orbit_rate)
        plt.figure(figsize=(10,5))
        sns.barplot(data=df, x="Orbit", y="Class", errorbar=None)
        plt.title("Observed landing success proportion by orbit")
        plt.xlabel("Orbit"); plt.ylabel("Mean landing class")
        plt.xticks(rotation=30); plt.tight_layout()
        plt.savefig("figures/success_by_orbit.png", dpi=180); plt.show()
    if "PayloadMass" in df.columns and "Class" in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=df, x="Class", y="PayloadMass")
        plt.title("Payload mass by landing class")
        plt.xlabel("Landing class"); plt.ylabel("Payload mass (kg)")
        plt.tight_layout(); plt.savefig("figures/payload_by_class.png", dpi=180); plt.show()
    if "FlightNumber" in df.columns and "Class" in df.columns:
        plt.figure(figsize=(8,5))
        sns.scatterplot(data=df, x="FlightNumber", y="Class", hue="Class", alpha=.75)
        plt.title("Flight number versus landing class")
        plt.tight_layout(); plt.savefig("figures/flight_number_class.png", dpi=180); plt.show()


## 4B. Yearly launch trend
The rubric asks for yearly trends. This cell uses the date column if it is present and produces a chart based on the loaded course data.


In [ ]:
if df is not None and "Date" in df.columns:
    yearly = df.copy()
    yearly["Date"] = pd.to_datetime(yearly["Date"], errors="coerce")
    yearly = yearly.dropna(subset=["Date"])
    yearly["Year"] = yearly["Date"].dt.year
    yearly_counts = yearly.groupby("Year").size().rename("Launches").reset_index()
    display(yearly_counts)
    plt.figure(figsize=(9,5))
    sns.lineplot(data=yearly_counts, x="Year", y="Launches", marker="o")
    plt.title("Number of launches by year")
    plt.xlabel("Year"); plt.ylabel("Launches")
    plt.tight_layout(); plt.savefig("figures/yearly_launch_trend.png", dpi=180); plt.show()
else:
    print("A Date column is required for the yearly trend. Confirm the course dataset columns.")


## 5. SQL-style analysis using SQLite
This reproduces common EDA SQL tasks locally, without requiring an external database service.

In [ ]:
import sqlite3
if df is not None:
    sql_df = df.copy()
    # SQLite can store datetime values as text.
    if "Date" in sql_df.columns:
        sql_df["Date"] = sql_df["Date"].astype(str)
    con = sqlite3.connect(":memory:")
    sql_df.to_sql("SPACEXTBL", con, index=False, if_exists="replace")
    queries = {
        "Unique launch sites": "SELECT DISTINCT LaunchSite FROM SPACEXTBL ORDER BY LaunchSite",
        "Launch counts by site": "SELECT LaunchSite, COUNT(*) AS Launch_Count FROM SPACEXTBL GROUP BY LaunchSite ORDER BY Launch_Count DESC",
        "Payload mass range": "SELECT MIN(PayloadMass) AS Minimum_Payload_Mass, MAX(PayloadMass) AS Maximum_Payload_Mass FROM SPACEXTBL",
        "Mission outcomes by launch site": "SELECT LaunchSite, Class, COUNT(*) AS Outcome_Count FROM SPACEXTBL GROUP BY LaunchSite, Class ORDER BY LaunchSite, Class"
    }
    for title, query in queries.items():
        print("\n", title, "\n", query)
        try:
            display(pd.read_sql_query(query, con))
        except Exception as exc:
            print("Query unavailable because a required column is absent:", exc)
    con.close()


## 6. Geospatial launch-site map
The course dataset includes site coordinates in some versions. This section creates a Folium map if coordinates and Folium are available.

In [ ]:
if df is not None and {"Latitude","Longitude","LaunchSite"}.issubset(df.columns):
    try:
        import folium
        from folium.plugins import MarkerCluster
        geo = df[["LaunchSite","Latitude","Longitude"]].dropna().drop_duplicates()
        center = [geo["Latitude"].mean(), geo["Longitude"].mean()]
        fmap = folium.Map(location=center, zoom_start=4, tiles="OpenStreetMap")
        cluster = MarkerCluster().add_to(fmap)
        for _, row in geo.iterrows():
            folium.Marker(
                location=[row["Latitude"], row["Longitude"]],
                tooltip=str(row["LaunchSite"]),
                popup=f"Launch site: {row['LaunchSite']}"
            ).add_to(cluster)
        fmap.save("figures/launch_sites_map.html")
        display(fmap)
        print("Saved interactive map to figures/launch_sites_map.html")
    except ImportError:
        print("Install folium with: pip install folium")
else:
    print("Map requires Latitude, Longitude, and LaunchSite columns in the loaded dataset.")


## 7. Dashboard data preparation
The script `dashboard.py` in the package uses the same course CSV and provides a launch-site dropdown and payload slider. Run it separately after installing Dash.

In [ ]:
if df is not None:
    dash_columns = [c for c in ["FlightNumber","LaunchSite","PayloadMass","Class"] if c in df.columns]
    if len(dash_columns) == 4:
        df[dash_columns].to_csv("spacex_launch_dash.csv", index=False)
        print("Saved dashboard data: spacex_launch_dash.csv")


## 8. Predictive analysis
Use the course-provided `Class` target and selected numeric/categorical features. The preprocessing pipeline avoids fitting the scaler on the test data. Model scores are generated from this execution; do not copy scores from another run.

In [ ]:
if df is not None and "Class" in df.columns:
    target = "Class"
    candidate_features = [c for c in ["FlightNumber","PayloadMass","Orbit","LaunchSite","Flights","GridFins","Reused","Legs","Block","ReusedCount","Serial"] if c in df.columns]
    model_df = df[candidate_features + [target]].copy()
    model_df = model_df.dropna(subset=[target])
    X = model_df[candidate_features]
    y = model_df[target].astype(int)
    X = pd.get_dummies(X, columns=[c for c in X.select_dtypes(include=["object","category","bool"]).columns], dummy_na=True)
    X = X.replace([np.inf,-np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True)).fillna(0)
    stratify = y if y.nunique() > 1 and y.value_counts().min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=stratify
    )
    models = {
        "Logistic Regression": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=3000, random_state=SEED))]),
        "SVM (RBF)": Pipeline([("scale", StandardScaler()), ("model", SVC(kernel="rbf", random_state=SEED))]),
        "Decision Tree": DecisionTreeClassifier(random_state=SEED, max_depth=5),
        "KNN": Pipeline([("scale", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=5))])
    }
    rows, fitted = [], {}
    for name, estimator in models.items():
        estimator.fit(X_train, y_train)
        pred = estimator.predict(X_test)
        fitted[name] = estimator
        rows.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall": recall_score(y_test, pred, zero_division=0),
            "F1": f1_score(y_test, pred, zero_division=0)
        })
    results = pd.DataFrame(rows).sort_values("F1", ascending=False)
    display(results.round(3))
    best_name = results.iloc[0]["Model"]
    best_model = fitted[best_name]
    best_pred = best_model.predict(X_test)
    print("Best model by test-set F1 in this run:", best_name)
    print(classification_report(y_test, best_pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, best_pred)
    plt.title(f"Confusion matrix — {best_name}")
    plt.tight_layout(); plt.savefig("figures/confusion_matrix.png", dpi=180); plt.show()
    results.to_csv("model_comparison_results.csv", index=False)
    print("Saved model metrics to model_comparison_results.csv")
else:
    print("Load the course dataset and confirm it contains the binary Class target before running model training.")


## 9. Conclusions — complete after execution
Use the generated tables and plots to write 3–5 evidence-based findings. Include:
- which launch sites/orbits have higher observed landing success proportions and the sample sizes;
- whether payload mass and flight number show visible associations with landing class;
- the best-performing model according to the metric you chose, including its measured test-set scores;
- a limitation (small historical dataset, omitted variables, or possible time-related drift);
- a business implication and a caution that predictions are estimates, not guarantees.

**Do not leave this section generic:** cite specific values from the outputs above. Save the notebook after all cells have run and verify that charts and tables are visible in the saved file.


## 10. References
- IBM/Coursera, *Applied Data Science Capstone*: https://www.coursera.org/learn/applied-data-science-capstone
- IBM Skills Network course dataset, `dataset_part_2.csv` (loaded from the course data endpoint above).
- SpaceX API repository: https://github.com/r-spacex/SpaceX-API
- pandas documentation: https://pandas.pydata.org/docs/
- scikit-learn documentation: https://scikit-learn.org/stable/
- Plotly Dash documentation: https://dash.plotly.com/
- Folium documentation: https://python-visualization.github.io/folium/latest/
